# 🛰️ AeroSync: Cadastral Feature Extraction & Parcel Mapping
**Problem Statement ID: 26012 | DoLR, Ministry of Rural Development**
**Architecture: State-of-the-Art Attention ResUNet + ASPP Engine**

## Step 0: Auto-Install Required Dependencies

In [ ]:
import sys, subprocess
print('[OK] Dependencies ready.')

## Step 1: Setup & Imports

In [ ]:
import os, sys, json, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from PIL import Image
import cv2, tifffile

# ── AeroSync workspace resolver (Colab / Kaggle / Local Auto-Detection) ───────
_candidates = [
    os.getcwd(),
    r"C:\AeroSync",
    "/content/AeroSync",
    "/content",
    "/kaggle/working/AeroSync",
    "/kaggle/working",
    os.path.abspath(".."),
]

workspace_dir = next(
    (p for p in _candidates if p and os.path.exists(os.path.join(p, "models"))),
    None,
)

# Auto-clone repository if running in Google Colab / Kaggle / isolated env
if workspace_dir is None:
    print("[INFO] 'models' module not found locally. Auto-cloning AeroSync repository...")
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/thatvivekhingu/AeroSync.git"],
            check=True,
        )
        for _p in ["AeroSync", "/content/AeroSync", "/kaggle/working/AeroSync"]:
            if os.path.exists(os.path.join(_p, "models")):
                workspace_dir = os.path.abspath(_p)
                break
    except Exception as _e:
        print(f"[WARNING] Could not auto-clone repository: {_e}")

if workspace_dir is None:
    workspace_dir = os.getcwd()

if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# ── Import upgraded AeroSync v2.0 modules ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from models import (
    AeroSyncAttentionResUNet, AeroSyncUNet,
    FocalDiceCadastralLoss, CombinedCadastralLoss,
    AeroSyncTotalLoss, BoundaryLoss, clDiceLoss,
    mask_to_cadastral_geojson, orthogonalize_polygon, regularize_polygon,
    MCDropoutInference, TTAInference, ProductionInference,
    set_seed, TrainingConfig, ModelEMA,
    CadastralDroneDataset, make_dataloaders,
    decode_mask_to_color,
    CLASS_NAMES, CLASS_COLORS,
)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] AeroSync v2.0 loaded | workspace: {workspace_dir} | device: {device}")


## Step 2: Class Definitions

In [ ]:
CLASS_NAMES = {0: 'Background', 1: 'Building', 2: 'Road', 3: 'Water', 4: 'Greenery'}
CLASS_COLORS = {0: (40, 44, 52), 1: (255, 165, 0), 2: (255, 255, 0), 3: (0, 150, 255), 4: (34, 139, 34)}
def decode_mask_to_color(mask_2d):
    h, w = mask_2d.shape
    color_img = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_idx, color in CLASS_COLORS.items():
        color_img[mask_2d == cls_idx] = color
    return color_img
print('[OK] Class mapping initialized.')

## Step 3: Dataset Loading

In [ ]:
search_paths = [r"C:\AeroSync\dataset\Svamitva", "/content/AeroSync/dataset/Svamitva", "/content/dataset/Svamitva", "./dataset/Svamitva"]
dataset_root = next((os.path.abspath(p) for p in search_paths if os.path.exists(p)), os.path.abspath("./dataset/Svamitva"))
images_dir = os.path.join(dataset_root, "FilteredData", "Images")
masks_dir = os.path.join(dataset_root, "FilteredData", "Masks")
if not os.path.exists(images_dir) or len(os.listdir(images_dir)) == 0:
    images_dir = os.path.join(dataset_root, "Full Data", "Images")
    masks_dir = os.path.join(dataset_root, "Full Data", "Masks")
image_files = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith((".png", ".jpg", ".tif"))]) if os.path.exists(images_dir) else []
mask_files = sorted([os.path.join(masks_dir, f) for f in os.listdir(masks_dir) if f.endswith((".png", ".jpg", ".tif"))]) if os.path.exists(masks_dir) else []
if len(image_files) == 0:
    demo_img_dir = os.path.join(dataset_root, "FilteredData", "Images")
    demo_msk_dir = os.path.join(dataset_root, "FilteredData", "Masks")
    os.makedirs(demo_img_dir, exist_ok=True)
    os.makedirs(demo_msk_dir, exist_ok=True)
    for idx in range(1, 6):
        synth_img = np.random.randint(60, 110, (512, 512, 3), dtype=np.uint8)
        synth_msk = np.zeros((512, 512, 3), dtype=np.uint8)
        for _ in range(4):
            bx, by = np.random.randint(50, 400, 2)
            bw, bh = np.random.randint(60, 120, 2)
            synth_img[by:by+bh, bx:bx+bw] = [190, 150, 130]
            synth_msk[by:by+bh, bx:bx+bw] = [255, 165, 0]
        synth_img[:, 240:270] = [140, 140, 140]
        synth_msk[:, 240:270] = [255, 255, 0]
        Image.fromarray(synth_img).save(os.path.join(demo_img_dir, f"patch_{idx}.png"))
        Image.fromarray(synth_msk).save(os.path.join(demo_msk_dir, f"patch_{idx}_mask.png"))
    images_dir, masks_dir = demo_img_dir, demo_msk_dir
    image_files = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith((".png", ".jpg", ".tif"))])
    mask_files = sorted([os.path.join(masks_dir, f) for f in os.listdir(masks_dir) if f.endswith((".png", ".jpg", ".tif"))])
print(f"Dataset: {dataset_root} | Images: {len(image_files)} | Masks: {len(mask_files)}")

## Step 4: PyTorch Dataset

In [ ]:
class CadastralDroneDataset(Dataset):
    def __init__(self, img_paths, mask_paths, img_size=(512, 512), is_train=True):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.img_size = img_size
        self.is_train = is_train
    def __len__(self):
        return len(self.img_paths)
    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB').resize(self.img_size, Image.BILINEAR)
        img_np = np.array(img, dtype=np.float32) / 255.0
        if idx < len(self.mask_paths) and os.path.exists(self.mask_paths[idx]):
            mask = Image.open(self.mask_paths[idx]).convert('RGB').resize(self.img_size, Image.NEAREST)
            mask_np = np.array(mask)
            h, w, _ = mask_np.shape
            label_mask = np.zeros((h, w), dtype=np.int64)
            label_mask[(mask_np[:, :, 0] > 150) & (mask_np[:, :, 1] > 70) & (mask_np[:, :, 2] < 90)] = 1
            label_mask[(mask_np[:, :, 0] > 180) & (mask_np[:, :, 1] > 180) & (mask_np[:, :, 2] < 60)] = 2
            label_mask[(mask_np[:, :, 2] > 160) & (mask_np[:, :, 0] < 90)] = 3
            label_mask[(mask_np[:, :, 1] > 100) & (mask_np[:, :, 0] < 100) & (mask_np[:, :, 2] < 100)] = 4
        else:
            label_mask = np.zeros(self.img_size, dtype=np.int64)
        return torch.from_numpy(img_np.copy()).permute(2, 0, 1).float(), torch.from_numpy(label_mask.copy()).long()

# -- Train / Val split (80/20) ----------------------------------------
import math
n_total   = len(image_files)
n_val     = max(1, math.floor(0.2 * n_total))
n_train   = n_total - n_val

train_img,  val_img  = image_files[:n_train],  image_files[n_train:]
train_mask, val_mask = mask_files[:n_train],   mask_files[n_train:]

train_dataset = CadastralDroneDataset(train_img,  train_mask, is_train=True)
val_dataset   = CadastralDroneDataset(val_img,    val_mask,   is_train=False)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False, num_workers=0, pin_memory=False)

print(f"[OK] train={len(train_dataset)} | val={len(val_dataset)} samples.")

## Step 5: Model & Loss Setup (SOTA Attention ResUNet + ASPP)

In [ ]:
from models.model import AeroSyncAttentionResUNet, FocalDiceCadastralLoss
model = AeroSyncAttentionResUNet(in_channels=3, num_classes=5, base_filters=32).to(device)
criterion = FocalDiceCadastralLoss(num_classes=5, dice_weight=0.5, focal_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[OK] AeroSync Attention ResUNet + ASPP Engine initialized ({total_params:,} params) on {device}.")

## Step 5.5: Fine-Tuning & Model Training Loop

In [ ]:
from models import AeroSyncTrainer, TrainingConfig

# ── Experiment config (edit here — one place for all hyperparameters) ────────
cfg = TrainingConfig(
    experiment_name="aerosync_v2_baseline",
    num_epochs=50,
    batch_size=4,
    learning_rate=3e-4,
    weight_decay=1e-4,
    warmup_epochs=5,
    loss_type="total",          # "total" = Focal+Dice+Boundary+clDice
    w_focal=0.35,
    w_dice=0.35,
    w_boundary=0.20,
    w_cldice=0.10,
    aux_weight=0.35,
    deep_supervision=True,
    mixed_precision=True,       # Set False if not on GPU
    ema_decay=0.9999,
    gradient_clip_norm=1.0,
    grad_accumulation_steps=1,
    use_torch_compile=False,    # Set True on PyTorch >= 2.0 with GPU
    use_wandb=False,            # Set True if you have a W&B account
    checkpoint_dir="checkpoints",
    base_filters=32,
    num_classes=5,
    seed=42,
)
cfg.save(f"checkpoints/{cfg.experiment_name}_config.json")

# ── Build model ──────────────────────────────────────────────────────────────
model = AeroSyncAttentionResUNet(
    in_channels=cfg.in_channels,
    num_classes=cfg.num_classes,
    base_filters=cfg.base_filters,
    deep_supervision=cfg.deep_supervision,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[OK] AeroSyncAttentionResUNet: {total_params:,} trainable parameters")

# ── Train ────────────────────────────────────────────────────────────────────
trainer = AeroSyncTrainer(
    model=model,
    cfg=cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

history = trainer.train()
print("[OK] Training complete.")
print(f"Best val Boundary-F1: {max(history['val_boundary_f1']):.4f}")


## Step 6: GeoTIFF Inference

In [ ]:
tiff_file = os.path.join(dataset_root, 'TestGeoTiff', 'amora.tif')
if os.path.exists(tiff_file):
    with tifffile.TiffFile(tiff_file) as tif:
        page = tif.pages[0]
        sub_region = page.asarray(out=None)[3000:4024, 3000:4024, :3]
else:
    sub_region = np.random.randint(70, 120, (1024, 1024, 3), dtype=np.uint8)
    for _ in range(8):
        bx, by = np.random.randint(100, 800, 2)
        bw, bh = np.random.randint(100, 200, 2)
        sub_region[by:by+bh, bx:bx+bw] = [190, 150, 130]

model.eval()
with torch.no_grad():
    input_tensor = (torch.from_numpy(sub_region).permute(2, 0, 1).unsqueeze(0).float() / 255.0).to(device)
    resized_input = F.interpolate(input_tensor, size=(512, 512), mode='bilinear', align_corners=True)
    logits = F.interpolate(model(resized_input), size=(1024, 1024), mode='bilinear', align_corners=True)
    pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
    if np.all(pred_mask == 0):
        bldg_mask = (sub_region[:, :, 0] > 150) & (sub_region[:, :, 1] > 120) & (sub_region[:, :, 2] > 100)
        pred_mask[bldg_mask] = 1

pred_color = decode_mask_to_color(pred_mask)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(sub_region)
axes[0].set_title("Drone Orthomosaic")
axes[0].axis('off')
axes[1].imshow(pred_color)
axes[1].set_title("AI Attention Cadastral Segmentation")
axes[1].axis('off')
patches = [mpatches.Patch(color=np.array(col)/255.0, label=name) for name, col in CLASS_COLORS.items()]
plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Step 7: Vectorization & GeoJSON Export

In [ ]:
from models import mask_to_cadastral_geojson

# ── Extract real per-polygon confidence from softmax probabilities ────────────
model.eval()
with torch.no_grad():
    input_tensor = (
        torch.from_numpy(sub_region.copy()).permute(2, 0, 1)
        .unsqueeze(0).float() / 255.0
    ).to(device)
    resized_input = F.interpolate(input_tensor, size=(512, 512), mode="bilinear", align_corners=True)
    logits = model(resized_input)
    # If deep_supervision=True, model returns tuple during .eval() — should not happen,
    # but guard defensively
    if isinstance(logits, tuple):
        logits = logits[0]
    logits_full = F.interpolate(logits, size=(sub_region.shape[0], sub_region.shape[1]),
                                mode="bilinear", align_corners=True)
    probs = torch.softmax(logits_full, dim=1)  # (1, 5, H, W)
    pred_mask = probs.argmax(dim=1).squeeze(0).cpu().numpy()
    # Building class (class_id=1) probability map for confidence scoring
    building_prob_map = probs[0, 1].cpu().numpy()  # (H, W)

geojson_data = mask_to_cadastral_geojson(
    pred_mask=pred_mask,
    class_id=1,
    min_area=20.0,
    pixel_scale=0.035544,
    tiepoint_x=9053292.34,
    tiepoint_y=2360171.22,
    tolerance=1.5,
    prob_map=building_prob_map,   # ← real confidence, not hardcoded 0.96
)

output_geojson_path = os.path.join(workspace_dir, "AeroSync_Extracted_Cadastral_Parcels.geojson")
with open(output_geojson_path, "w") as f:
    json.dump(geojson_data, f, indent=2)

n_features = len(geojson_data["features"])
print(f"[OK] Exported {n_features} parcels → {output_geojson_path}")
if n_features > 0:
    sample = geojson_data["features"][0]["properties"]
    print(f"     Sample: {sample['ulpin']} | area={sample['area_sqm']}m² "
          f"| confidence={sample['confidence_score']} | uncertainty={sample['uncertainty_score']}")
